In [75]:
from dotenv import load_dotenv

In [2]:
load_dotenv() 

True

In [6]:
%pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [37]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

Step 1a - Indexing (Document Ingestion)

In [99]:
video_id = "TjPPIf-M6KE"  # only the ID, not full URL

try:
    ytt_api = YouTubeTranscriptApi()
    fetched = ytt_api.fetch(video_id, languages=["en"])

    # Flatten to plain text
    transcript = " ".join(snippet.text for snippet in fetched)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

So, what version of you has died? Like, who were you when you started and who have you become? >> Um, I was like much more carefree. Like, when we started off, I was just like, you know, I was like I had a very easy [music] life at the time. I had a lot of hobbies and, you know, now I'm like incredibly [music] precisely scheduled and I have not gotten to do anything spontaneous in a long time. >> Was there any point ever in your life where nobody else was convinced [music] but just you? >> Launching ChatGPT was kind of an example of that. Not that people didn't want to launch a chatbot, but most people thought it was too early and not good enough. >> When you talk to other people, do they tell you that you're delusional sometime? >> Yes, but I think I should probably get more of that. >> Why? >> Well, one way that like a lot of people in roles like mine have had things go wrong is they get surrounded by too many yes men. >> You've gone through like people suing you, people dropping out

Step 1b - Indexing (Text Splitting)

In [100]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [101]:
len(chunks)

80

In [102]:
chunks[10]

Document(metadata={}, page_content="is it Why there's difference? >> Um One thing that many people have observed is that countries that are developing faster and where the economy is growing faster and there's like more of a need for products and services and like, you know, there's just huge under supply are more welcoming of AI than more developed economies. So, I'm sure that's part of it. >> Interesting. And the other thing when I when I read about you, right? Like with I'm and I read about you in US and I read about you in India. This sentiment of being pessimistic and optimist, it also reflects about you. And that's the most fascinating thing that you're this person who started this most remarkable technological revolution here of our age. You were the first guy. You don't even own equity in your own company. And yet, all of a sudden you became the bad guy. Like how? Why? >> Um Well, I think the US has great skepticism of people in power and in particular of like the just enormous

In [15]:
%pip install -q langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [16]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [103]:
import time

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

batch_size = 20
vector_store = None

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    if vector_store is None:
        vector_store = FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)
    print(f"Embedded {min(i + batch_size, len(chunks))}/{len(chunks)}")
    time.sleep(15)

print(vector_store.index.ntotal)  

Embedded 20/80
Embedded 40/80
Embedded 60/80
Embedded 80/80
80


In [104]:
vector_store.index_to_docstore_id

{0: 'cd50d7ac-0a01-4ee4-a191-f313448f0f3a',
 1: '93faf123-6d7c-402a-994d-a56f71bebb99',
 2: 'e1681952-3c16-4e04-81c4-0108dbc15988',
 3: '3cf8a44c-70f4-4314-b842-75f2f9276bdf',
 4: '942d5a5a-bfe5-4a2f-8bd4-8c924b0c9442',
 5: '409ef24d-cd56-47cf-af9a-3b5d396a05af',
 6: 'f96487f9-8362-40bf-b956-d2e4bc375d61',
 7: '23a6dc31-62ae-4223-81d6-f720fc2703af',
 8: '6e7fee94-e935-40cc-a72f-fae248403fb8',
 9: '9f555cd0-65ad-43da-8f63-0a70263d578d',
 10: '5cd46b1e-e8b1-4d24-a189-7dd733fab277',
 11: '6fed70a1-22aa-427b-be82-c333134715bf',
 12: '32020f45-925a-42ba-a928-3af8995707b3',
 13: '23fd839e-b2b8-48fd-8b9e-e09fb96d618e',
 14: '0daffdf2-8f4b-4b32-83d9-0acbdaa937a5',
 15: '5ba23c09-737c-476c-a781-5a2b3a70da1d',
 16: '88a90cec-8d0b-4b83-b8cf-b62fa9e71b6a',
 17: '2aae1b53-3fc3-4999-9a8a-dd6a5be098e3',
 18: '1d36c9d0-3ea0-4df3-b7bb-8ad16e7cf98f',
 19: '23252955-e079-49b2-b0bd-5b220c57f652',
 20: '2955b04c-09f3-49ec-9e01-3d1851773a2c',
 21: 'bdc45676-188f-4bdb-9d7e-70edcd4a1d31',
 22: '59a63694-d4e3-

Step 2 - Retrieval

In [105]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs={"k": 4})

In [106]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7011bb3890>, search_kwargs={'k': 4})

In [107]:
retriever.invoke('What is the main topic of the video?')

[Document(id='409ef24d-cd56-47cf-af9a-3b5d396a05af', metadata={}, page_content="appreciate you doing this. >> You know, if someone's watching you for the first time and they have no clue who you are, and how would you explain them who you are in next 60 seconds? >> Um I'm like a tech fan, like a curious person about the world, and a believer that AI is going to be this thing that touches all of our lives, hopefully in mostly super positive ways. Um I'm a dad. I really love my kids. That's what I would say. >> But do you ever think about certain things? Like this is how you going to think or say or speak about? >> Um I do just kind of try to naturally talk about what's on my mind, uh or what I'm interested in. >> Okay. And were you like this when you were a kid? Always thinking about what interesting to say next. >> I think I was like that as a kid. I think I would just try to like I I think I just always like followed what I was curious about. >> And do you, as a kid, did you ever feel

Step 3 - Augmentation

In [108]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.2, max_output_tokens=4096)

In [109]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [110]:
question          = "What is the main topic of the video?"
retrieved_docs    = retriever.invoke(question)

In [111]:
retrieved_docs

[Document(id='409ef24d-cd56-47cf-af9a-3b5d396a05af', metadata={}, page_content="appreciate you doing this. >> You know, if someone's watching you for the first time and they have no clue who you are, and how would you explain them who you are in next 60 seconds? >> Um I'm like a tech fan, like a curious person about the world, and a believer that AI is going to be this thing that touches all of our lives, hopefully in mostly super positive ways. Um I'm a dad. I really love my kids. That's what I would say. >> But do you ever think about certain things? Like this is how you going to think or say or speak about? >> Um I do just kind of try to naturally talk about what's on my mind, uh or what I'm interested in. >> Okay. And were you like this when you were a kid? Always thinking about what interesting to say next. >> I think I was like that as a kid. I think I would just try to like I I think I just always like followed what I was curious about. >> And do you, as a kid, did you ever feel

In [112]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'appreciate you doing this. >> You know, if someone\'s watching you for the first time and they have no clue who you are, and how would you explain them who you are in next 60 seconds? >> Um I\'m like a tech fan, like a curious person about the world, and a believer that AI is going to be this thing that touches all of our lives, hopefully in mostly super positive ways. Um I\'m a dad. I really love my kids. That\'s what I would say. >> But do you ever think about certain things? Like this is how you going to think or say or speak about? >> Um I do just kind of try to naturally talk about what\'s on my mind, uh or what I\'m interested in. >> Okay. And were you like this when you were a kid? Always thinking about what interesting to say next. >> I think I was like that as a kid. I think I would just try to like I I think I just always like followed what I was curious about. >> And do you, as a kid, did you ever feel inferior to someone? >> All the time. I mean, I\'m sure most kids do. Uh

In [113]:
final_prompt = prompt.format(context=context_text, question=question)

Step 4 - Generation

In [114]:
answer = llm.invoke(final_prompt)
print(answer.text)

/opt/anaconda3/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the transcript context, the video focuses on understanding Sam Altman's mind and how he thinks. Specifically, it covers:

* How he thinks and makes decisions.
* His childhood, insecurities, and fears.
* How he views himself as a friend, a father, and a builder of technology/AI that will shape humanity.
* How he notices patterns and identifies non-obvious things to work on.


Building a Chain

In [68]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [115]:
def format_docs(retrieved_docs):
    context_text =  "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [116]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [118]:
parallel_chain.invoke('who is the speaker in this video?')

{'context': "appreciate you doing this. >> You know, if someone's watching you for the first time and they have no clue who you are, and how would you explain them who you are in next 60 seconds? >> Um I'm like a tech fan, like a curious person about the world, and a believer that AI is going to be this thing that touches all of our lives, hopefully in mostly super positive ways. Um I'm a dad. I really love my kids. That's what I would say. >> But do you ever think about certain things? Like this is how you going to think or say or speak about? >> Um I do just kind of try to naturally talk about what's on my mind, uh or what I'm interested in. >> Okay. And were you like this when you were a kid? Always thinking about what interesting to say next. >> I think I was like that as a kid. I think I would just try to like I I think I just always like followed what I was curious about. >> And do you, as a kid, did you ever feel inferior to someone? >> All the time. I mean, I'm sure most kids d

In [119]:
parser = StrOutputParser()

In [121]:
main_chain = parallel_chain | prompt | llm | parser

In [122]:
main_chain.invoke('Can you summarize the video')

/opt/anaconda3/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided transcript, the video features an episode of the podcast *Figuring Out* hosted by Raj, featuring guest Sam Altman. Here is a summary of the transcript contents:\n\n* **Host\'s Introduction & Goal:** The host, Raj, introduces Sam Altman and states that the podcast aims to explore Sam\'s mindset—his fears, insecurities, how he thinks as a friend and father, and how he identifies non-obvious patterns to build technology that will shape humanity. \n* **Sam Altman\'s Self-Description:** When asked to introduce himself, Sam describes himself as a tech fan, a curious person, a dad who loves his kids, and a believer that AI will touch everyone\'s lives in mostly positive ways.\n* **Reflections on Mindset and Childhood:** Sam explains that he naturally speaks about what is on his mind. He notes that as a kid, he always followed his curiosity and frequently felt inferior, which he assumes most kids experience.\n* **Behind-the-Scenes & Casual Interaction:** The host and gue